# Financial Fraud Detection - SageMaker Endpoint Testing

This notebook tests the deployed SageMaker endpoint running NVIDIA Triton for fraud detection inference.

The model (`prediction_and_shapley`) accepts merchant features, user features, and graph structure as inputs.
It returns fraud probability plus Shapley values for explainability.

## Setup

In [ ]:
import json
import numpy as np
import boto3

# Configuration
ENDPOINT_NAME = "fraud-detection-endpoint"  # Change to your endpoint name
AWS_REGION = "us-east-1"
AWS_PROFILE = None  # Set to profile name if needed, e.g., "zjacobso+nvidia-Admin"

In [ ]:
# Create SageMaker runtime client
if AWS_PROFILE:
    session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
else:
    session = boto3.Session(region_name=AWS_REGION)

runtime = session.client("sagemaker-runtime")

## Helper Functions

In [ ]:
def make_example_request(num_merchants=5, num_users=7, num_edges=3):
    """
    Create example inference request with random data.
    
    This mirrors the NVIDIA blueprint's test data structure.
    """
    merchant_feature_dim = 24
    user_feature_dim = 13
    user_to_merchant_feature_dim = 38

    # Node features
    x_merchant = np.random.randn(num_merchants, merchant_feature_dim).astype(np.float32)
    x_user = np.random.randn(num_users, user_feature_dim).astype(np.float32)

    # Shapley computation flag and feature masks
    compute_shap = np.array([True], dtype=np.bool_)
    feature_mask_merchant = np.random.randint(0, 2, size=(merchant_feature_dim,), dtype=np.int32)
    feature_mask_user = np.random.randint(0, 2, size=(user_feature_dim,), dtype=np.int32)

    # Edge index and attributes
    edge_index_user_to_merchant = np.vstack([
        np.random.randint(0, num_users, size=(num_edges,)),
        np.random.randint(0, num_merchants, size=(num_edges,))
    ]).astype(np.int64)
    
    edge_attr_user_to_merchant = np.random.randn(num_edges, user_to_merchant_feature_dim).astype(np.float32)
    feature_mask_user_to_merchant = np.random.randint(0, 2, size=(user_to_merchant_feature_dim,), dtype=np.int32)

    return {
        "x_merchant": x_merchant,
        "x_user": x_user,
        "COMPUTE_SHAP": compute_shap,
        "feature_mask_merchant": feature_mask_merchant,
        "feature_mask_user": feature_mask_user,
        "edge_index_user_to_merchant": edge_index_user_to_merchant,
        "edge_attr_user_to_merchant": edge_attr_user_to_merchant,
        "edge_feature_mask_user_to_merchant": feature_mask_user_to_merchant
    }

In [ ]:
def numpy_to_triton_input(data):
    """
    Convert numpy arrays to Triton inference request format.
    """
    inputs = []
    
    dtype_map = {
        np.float32: "FP32",
        np.float64: "FP64",
        np.int32: "INT32",
        np.int64: "INT64",
        np.bool_: "BOOL"
    }
    
    for name, arr in data.items():
        dtype = dtype_map.get(arr.dtype.type, "FP32")
        inputs.append({
            "name": name,
            "shape": list(arr.shape),
            "datatype": dtype,
            "data": arr.flatten().tolist()
        })
    
    return inputs

In [ ]:
def invoke_endpoint(data, compute_shap=False):
    """
    Send inference request to SageMaker endpoint.
    """
    # Override COMPUTE_SHAP flag
    data = data.copy()
    data["COMPUTE_SHAP"] = np.array([compute_shap], dtype=np.bool_)
    
    # Build Triton request
    request = {
        "inputs": numpy_to_triton_input(data),
        "outputs": [{"name": "PREDICTION"}]
    }
    
    # Add Shapley outputs if requested
    if compute_shap:
        for key in data:
            if key.startswith("x_"):
                node = key[len("x_"):]
                request["outputs"].append({"name": f"shap_values_{node}"})
            elif key.startswith("edge_attr_"):
                edge_name = key[len("edge_attr_"):]
                request["outputs"].append({"name": f"shap_values_{edge_name}"})
    
    # Invoke endpoint
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(request)
    )
    
    result = json.loads(response["Body"].read().decode())
    return result

In [ ]:
def parse_triton_response(response):
    """
    Parse Triton response into numpy arrays.
    """
    result = {}
    for output in response.get("outputs", []):
        name = output["name"]
        shape = output["shape"]
        data = output["data"]
        result[name] = np.array(data).reshape(shape)
    return result

## Test 1: Basic Health Check

In [ ]:
# Check endpoint status
sm = session.client("sagemaker")
endpoint_info = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Status: {endpoint_info['EndpointStatus']}")
print(f"Instance Type: {endpoint_info.get('ProductionVariants', [{}])[0].get('CurrentInstanceCount', 'N/A')} instance(s)")

## Test 2: Inference with Random Data

In [ ]:
# Create random test data
test_data = make_example_request(num_merchants=5, num_users=7, num_edges=3)

print("Input shapes:")
for name, arr in test_data.items():
    print(f"  {name}: {arr.shape} ({arr.dtype})")

In [ ]:
# Run inference (without Shapley values)
print("Running inference without Shapley values...")
response = invoke_endpoint(test_data, compute_shap=False)
result = parse_triton_response(response)

print(f"\nPredictions shape: {result['PREDICTION'].shape}")
print(f"Predictions: {result['PREDICTION'].flatten()}")

In [ ]:
# Run inference (with Shapley values)
print("Running inference with Shapley values...")
response = invoke_endpoint(test_data, compute_shap=True)
result = parse_triton_response(response)

print(f"\nOutputs:")
for name, arr in result.items():
    print(f"  {name}: {arr.shape}")

## Test 3: Performance Evaluation

To evaluate model performance, you need the preprocessed test data from the training pipeline.
Download from S3 or use locally if available.

In [ ]:
# Optional: Load real test data from S3
# Uncomment and modify paths as needed

# import os
# DATA_BUCKET = "fraud-detection-<account>-sm"
# TEST_DATA_PREFIX = "data/processed/gnn/test_gnn"
# LOCAL_TEST_DIR = "./test_data"
# 
# os.makedirs(LOCAL_TEST_DIR, exist_ok=True)
# s3 = session.client("s3")
# 
# # Download test data files
# for key in ["x_merchant.npy", "x_user.npy", "edge_index.npy", "edge_attr.npy", "y.npy"]:
#     s3.download_file(DATA_BUCKET, f"{TEST_DATA_PREFIX}/{key}", f"{LOCAL_TEST_DIR}/{key}")

In [ ]:
def compute_metrics(y_true, y_pred, threshold=0.5):
    """
    Compute classification metrics.
    """
    from sklearn.metrics import (
        confusion_matrix,
        accuracy_score,
        precision_score,
        recall_score,
        f1_score
    )
    
    y_pred_binary = (y_pred > threshold).astype(int)
    
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred_binary),
        "precision": precision_score(y_true, y_pred_binary, zero_division=0),
        "recall": recall_score(y_true, y_pred_binary, zero_division=0),
        "f1": f1_score(y_true, y_pred_binary, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred_binary)
    }
    
    return metrics

In [ ]:
def print_metrics(metrics):
    """
    Pretty print classification metrics.
    """
    print("\n" + "="*50)
    print("Model Performance Metrics")
    print("="*50)
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1 Score:  {metrics['f1']:.4f}")
    print("\nConfusion Matrix:")
    print("                 Predicted")
    print("               Non-Fraud  Fraud")
    cm = metrics['confusion_matrix']
    print(f"Actual Non-Fraud  {cm[0,0]:>6}  {cm[0,1]:>6}")
    print(f"       Fraud      {cm[1,0]:>6}  {cm[1,1]:>6}")
    print("="*50)

## Test 4: Latency Benchmark

In [ ]:
import time

def benchmark_latency(data, num_requests=10, compute_shap=False):
    """
    Benchmark inference latency.
    """
    latencies = []
    
    for i in range(num_requests):
        start = time.time()
        response = invoke_endpoint(data, compute_shap=compute_shap)
        end = time.time()
        latencies.append((end - start) * 1000)  # Convert to ms
        
    return {
        "mean_ms": np.mean(latencies),
        "std_ms": np.std(latencies),
        "min_ms": np.min(latencies),
        "max_ms": np.max(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
    }

In [ ]:
# Benchmark without Shapley values
print("Benchmarking inference latency (without Shapley)...")
latency_stats = benchmark_latency(test_data, num_requests=10, compute_shap=False)

print(f"\nLatency Statistics (10 requests):")
print(f"  Mean:   {latency_stats['mean_ms']:.1f} ms")
print(f"  Std:    {latency_stats['std_ms']:.1f} ms")
print(f"  Min:    {latency_stats['min_ms']:.1f} ms")
print(f"  Max:    {latency_stats['max_ms']:.1f} ms")
print(f"  P50:    {latency_stats['p50_ms']:.1f} ms")
print(f"  P95:    {latency_stats['p95_ms']:.1f} ms")
print(f"  P99:    {latency_stats['p99_ms']:.1f} ms")

In [ ]:
# Benchmark with Shapley values
print("Benchmarking inference latency (with Shapley)...")
latency_stats_shap = benchmark_latency(test_data, num_requests=10, compute_shap=True)

print(f"\nLatency Statistics with Shapley (10 requests):")
print(f"  Mean:   {latency_stats_shap['mean_ms']:.1f} ms")
print(f"  P50:    {latency_stats_shap['p50_ms']:.1f} ms")
print(f"  P95:    {latency_stats_shap['p95_ms']:.1f} ms")

## Cleanup

Don't forget to delete the endpoint when you're done testing to avoid charges:

```bash
aws sagemaker delete-endpoint --endpoint-name fraud-detection-endpoint
```

Or use the Makefile:

```bash
make clean-endpoints
```